In [ ]:
import sys
import time
import numpy as np
import pandas as pd
import glob as glob
import os
import itertools

from scipy.interpolate import interp1d

from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.wcs import WCS
from astropy.modeling import models, fitting
from astropy.modeling.fitting import LevMarLSQFitter
from astropy.table import Table, QTable, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord, angular_separation

from astropy.nddata.utils import Cutout2D

from scipy.interpolate import interp1d
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
from pydol.photometry.scripts.catalog_filter import box
import json

In [ ]:
%matplotlib inline
from matplotlib import style, pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker
import matplotlib.colors as col
from matplotlib.colors import ListedColormap
import seaborn as sb
import matplotlib.gridspec as gridspec
sb.set_style('white')
from matplotlib.ticker import (MultipleLocator, AutoLocator, AutoMinorLocator)

from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcParams['image.cmap'] = 'jet'
plt.rcParams['image.origin'] = 'lower'
plt.rcParams['figure.figsize'] = (7,5)
plt.rcParams['axes.titlesize'] = plt.rcParams['axes.labelsize'] = 35
plt.rcParams['xtick.labelsize'] = plt.rcParams['ytick.labelsize'] = 35

font1 = {'family': 'sans-serif', 'color': 'black', 'weight': 'normal', 'size': '15'}
font2 = {'family': 'sans-serif', 'color': 'black', 'weight': 'normal', 'size': '25'}

In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"]})


## **JWST**

In [ ]:
Av_dict = { 
            'f275w': 2.02499,
            'f336w': 1.67536,
            'f435w': 1.33879,
            'f555w': 1.03065,
            'f814w': 0.59696,
    
            'f090w': 0.583,
            'f115w': 0.419,
            'f150w': 0.287,
            'f200w': 0.195,
    
            'f438w': 1.34148,
            'f606w': 0.90941,
            'f814w': 0.59845
          }

with open('../data/DS9 regions/regions90_ngc628.json') as json_file:
    regions_dict = json.load(json_file)    

regions_dict.update({'ngc628' : {'ra'   : 24.1738983, 
                       'dec'  : 15.7836543,
                        'dismod' : 29.81,
                        'Av': 0.19},
    
            'm83' :   {'ra'   : 204.2536827 ,  
                       'dec'  : -29.8655432,                        
                       'dismod' : 28.41,
                        'Av': 0.18166},
    
            'm51' :  {'ra'   : 202.4696435,
                      'dec'  : 47.1952141,
                      'dismod' : 29.67,
                       'Av': 0.09331},
    
            'ngc4449' : {'ra' : 187.046349,
                         'dec' :44.093666,
                         'dismod' : 28.15,
                         'Av': 0.05208},
            'm82'     : {'ra' :  148.970500	  ,
                         'dec' : 69.679483}})

# **DOLPHOT Completeness**

In [ ]:
@models.custom_model
def pritchet(m, alpha=0.5, m_50=22):
    return 0.5*(1 - alpha*(m - m_50)/np.sqrt(1 + alpha**2*(m-m_50)**2))

@models.custom_model
def pritchet_inv(p, alpha=0.5, m_50=22):
    return m_50 + (1/alpha)*(1-2*p)/np.sqrt(1-(1-2*p)**2)

In [ ]:
# F115W
f115w = []
f115w_mags = []
f115w_f200w = []
for i in np.arange(20,30.5,0.5):
    fs = glob.glob("../photometry/ngc628/completeness/*{0:.1f}_20_20*_??.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_in = vstack(tabs)

    fs = glob.glob("../photometry/ngc628/completeness/*{0:.1f}_20_20*_??_filt.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_out = vstack(tabs)
    counts = []
    for j in range(90):
        ra_cen = regions_dict[f'reg_{j}']['ra']
        dec_cen = regions_dict[f'reg_{j}']['dec']
        
        tab_in = box(tabs_in, 'ra','dec', ra_cen, dec_cen, 0,0,24/3600,24/3600,131.67459)
        tab_out = box(tabs_out, 'ra','dec', ra_cen, dec_cen, 0,0,24/3600,24/3600,131.67459)
        tab_out = tab_out[abs(tab_out['mag_vega_F115W']-i)<=0.2]
        f115w_mags.append([[i]*len(tab_out),tab_out['mag_vega_F115W']])
        f115w_f200w.append([[i-20]*len(tab_out),tab_out['mag_vega_F115W']- tab_out['mag_vega_F200W']])

        if len(tab_out)!=0:
            counts.append(len(tab_out)/len(tab_in))
        else:
            counts.append(0)
        
    f115w.append(counts)
    
f115w = np.array(f115w)

f115w_m50 = []
f115w_a   = []
ms = np.arange(20,30.5,0.5)
init = pritchet()
fit = fitting.LevMarLSQFitter()
for i in range(90):
    y = f115w[:,i]
    model = fit(init, ms,y)
    f115w_m50.append(model.m_50.value)
    f115w_a.append(model.alpha.value)

f115w_m50 = np.array(f115w_m50).reshape(15,6)
f115w_a = np.array(f115w_a).reshape(15,6)

In [ ]:
 # F150W
f150w = []
for i in np.arange(20,30.5,0.5):
    fs = glob.glob("../photometry/ngc628/completeness/*20_{0:.1f}_20*_??.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_in = vstack(tabs)

    fs = glob.glob("../photometry/ngc628/completeness/*20_{0:.1f}_20*_??_filt.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_out = vstack(tabs)
    counts = []
    for j in range(90):
        ra_cen = regions_dict[f'reg_{j}']['ra']
        dec_cen = regions_dict[f'reg_{j}']['dec']
        
        tab_in = box(tabs_in, 'ra','dec', ra_cen, dec_cen, 0,0,24/3600,24/3600,131.67459)
        tab_out = box(tabs_out, 'ra','dec', ra_cen, dec_cen, 0,0,24/3600,24/3600,131.67459)
        tab_out = tab_out[abs(tab_out['mag_vega_F150W']-i)<=0.2]

        if len(tab_out)!=0:
            counts.append(len(tab_out)/len(tab_in))
        else:
            counts.append(0)
        
    f150w.append(counts)
    
f150w = np.array(f150w)

f150w_m50 = []
f150w_a   = []
ms = np.arange(20,30.5,0.5)
init = pritchet()
fit = fitting.LevMarLSQFitter()
for i in range(90):
    y = f150w[:,i]
    model = fit(init, ms,y)
    f150w_m50.append(model.m_50.value)
    f150w_a.append(model.alpha.value)

f150w_m50 = np.array(f150w_m50).reshape(15,6)
f150w_a = np.array(f150w_a).reshape(15,6)

In [ ]:
 # F200W
f200w = []
for i in np.arange(20,30.5,0.5):
    fs = glob.glob("../photometry/ngc628/completeness/*20_20_{0:.1f}*_??.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_in = vstack(tabs)

    fs = glob.glob("../photometry/ngc628/completeness/*20_20_{0:.1f}*_??_filt.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_out = vstack(tabs)
    counts = []
    for j in range(90):
        ra_cen = regions_dict[f'reg_{j}']['ra']
        dec_cen = regions_dict[f'reg_{j}']['dec']
        
        tab_in = box(tabs_in, 'ra','dec', ra_cen, dec_cen, 0,0,24/3600,24/3600,131.67459)
        tab_out = box(tabs_out, 'ra','dec', ra_cen, dec_cen, 0,0,24/3600,24/3600,131.67459)
        tab_out = tab_out[abs(tab_out['mag_vega_F200W']-i)<=0.2]

        if len(tab_out)!=0:
            counts.append(len(tab_out)/len(tab_in))
        else:
            counts.append(0)
        
    f200w.append(counts)
    
f200w = np.array(f200w)

f200w_m50 = []
f200w_a   = []
ms = np.arange(20,30.5,0.5)
init = pritchet()
fit = fitting.LevMarLSQFitter()
for i in range(90):
    y = f200w[:,i]
    model = fit(init, ms,y)
    f200w_m50.append(model.m_50.value)
    f200w_a.append(model.alpha.value)

f200w_m50 = np.array(f200w_m50).reshape(15,6)
f200w_a = np.array(f200w_a).reshape(15,6)

In [ ]:
coords = []
for i in range(90):
    coords.append([regions_dict[f'reg_{i}']['ra'], regions_dict[f'reg_{i}']['dec']])
    

In [ ]:
r = angular_separation(np.array(coords)[:,0]*u.deg, np.array(coords)[:,1]*u.deg, 
                        regions_dict['ngc628']['ra']*u.deg, regions_dict['ngc628']['dec']*u.deg).to(u.arcsec).value

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

init = models.Polynomial1D(3)
fit = fitting.LevMarLSQFitter()

rn = np.linspace(0,r.max())
ax.scatter(r.ravel(), f115w_m50.ravel(), color='blue')

model = fit(init, r.ravel(),f115w_m50.ravel())
ax.plot(rn, model(rn), color='blue')

ax.scatter(r.ravel(), f150w_m50.ravel(), color='green')

model = fit(init, r.ravel(),f150w_m50.ravel())
ax.plot(rn,  model(rn), color='green')

ax.scatter(r.ravel(), f200w_m50.ravel(), color='red')

model = fit(init, r.ravel(),f200w_m50.ravel())
ax.plot(rn,  model(rn), color='red')

ax.set_ylabel(r'$m_{50}$')
ax.set_xlabel(r'R (arcsec)')

ax.invert_yaxis()

ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

plt.show()

In [ ]:
np.save("../SFH/ngc628/F115W_m50.npy",f115w_m50)
np.save("../SFH/ngc628/F150W_m50.npy",f150w_m50)
np.save("../SFH/ngc628/F200W_m50.npy",f200w_m50)

np.save("../SFH/ngc628/F115W_a.npy",f115w_a)
np.save("../SFH/ngc628/F150W_a.npy",f150w_a)
np.save("../SFH/ngc628/F200W_a.npy",f200w_a)

# **M82**

In [ ]:
with open('../data/DS9 regions/regions_m82.json') as json_file:
    regions = json.load(json_file)  

regions_dict.update(regions)

In [ ]:
# F140M
F140M = []
for i in np.arange(20,30.5,0.5):
    fs = glob.glob("../photometry/m82/completeness/*{0:.1f}_20*_??.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_in = vstack(tabs)

    fs = glob.glob("../photometry/m82/completeness/*{0:.1f}_20*_??_filt.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_out = vstack(tabs)
    counts = []
    for j in range(60):
        ra_cen = regions_dict[f'reg_{j}']['ra']
        dec_cen = regions_dict[f'reg_{j}']['dec']
        
        tab_in = box(tabs_in, 'ra','dec', ra_cen, dec_cen, 0,0,30/3600,32/3600,320)
        tab_out = box(tabs_out, 'ra','dec', ra_cen, dec_cen, 0,0,30/3600,32/3600,320)
        tab_out = tab_out[abs(tab_out['mag_vega_F140M']-i)<=0.5]

        if len(tab_out)!=0:
            counts.append(len(tab_out)/len(tab_in))
        else:
            counts.append(0)
    F140M.append(counts)
    
F140M = np.array(F140M)

F140M_m50 = []
F140M_a   = []
ms = np.arange(20,30.5,0.5)
init = pritchet()
fit = fitting.LevMarLSQFitter()
for i in range(60):
    y = F140M[:,i]
    model = fit(init, ms,y)
    F140M_m50.append(model.m_50.value)
    F140M_a.append(model.alpha.value)

F140M_m50 = np.array(F140M_m50).reshape(10,6)
F140M_a = np.array(F140M_a).reshape(10,6)

In [ ]:
# F212N
F212N = []
for i in np.arange(20,30.5,0.5):
    fs = glob.glob("../photometry/m82/completeness/*20_{0:.1f}*_??.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_in = vstack(tabs)

    fs = glob.glob("../photometry/m82/completeness/*20_{0:.1f}*_??_filt.fits".format(i))
    tabs = []
    for f in fs:
        tabs.append(Table.read(f))
        
    tabs_out = vstack(tabs)
    counts = []
    for j in range(60):
        ra_cen = regions_dict[f'reg_{j}']['ra']
        dec_cen = regions_dict[f'reg_{j}']['dec']
        
        tab_in = box(tabs_in, 'ra','dec', ra_cen, dec_cen, 0,0,30/3600,32/3600,320)
        tab_out = box(tabs_out, 'ra','dec', ra_cen, dec_cen, 0,0,30/3600,32/3600,320)
        tab_out = tab_out[abs(tab_out['mag_vega_F212N']-i)<=0.5]

        if len(tab_out)!=0:
            counts.append(len(tab_out)/len(tab_in))
        else:
            counts.append(0)
        
    F212N.append(counts)
    
F212N = np.array(F212N)

F212N_m50 = []
F212N_a   = []
ms = np.arange(20,30.5,0.5)
init = pritchet()
fit = fitting.LevMarLSQFitter()
for i in range(60):
    y = F212N[:,i]
    model = fit(init, ms,y)
    F212N_m50.append(model.m_50.value)
    F212N_a.append(model.alpha.value)

F212N_m50 = np.array(F212N_m50).reshape(10,6)
F212N_a = np.array(F212N_a).reshape(10,6)

In [ ]:
coords = []
for i in range(60):
    coords.append([regions_dict[f'reg_{i}']['ra'], regions_dict[f'reg_{i}']['dec']])
    

In [ ]:
r = angular_separation(np.array(coords)[:,0]*u.deg, np.array(coords)[:,1]*u.deg, 
                        regions_dict['m82']['ra']*u.deg, regions_dict['m82']['dec']*u.deg).to(u.arcsec).value

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

init = models.Polynomial1D(3)
fit = fitting.LevMarLSQFitter()

rn = np.linspace(0,r.max())
ax.scatter(r.ravel(), F140M_m50.ravel(), color='blue')

ind = F140M_m50.ravel()>20
model = fit(init, r.ravel()[ind],F140M_m50.ravel()[ind])
ax.plot(rn, model(rn), color='blue')

ax.scatter(r.ravel(), F212N_m50.ravel(), color='red')

ind = F212N_m50.ravel()>20
model = fit(init, r.ravel()[ind],F212N_m50.ravel()[ind])
ax.plot(rn,  model(rn), color='red')

ax.set_ylabel(r'$m_{50}$')
ax.set_xlabel(r'R (arcsec)')

ax.invert_yaxis()

ax.xaxis.set_major_locator(AutoLocator())
ax.xaxis.set_minor_locator(AutoMinorLocator())

ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True, width=3)
ax.tick_params(which='minor', length=8, width=3)

plt.show()

In [ ]:
F140M_m50.T[0,0] = F140M_m50.T[5,9]
F140M_m50.T[1,0] = F140M_m50.T[4,9]

F140M_a.T[0,0] = F140M_a.T[5,9]
F140M_a.T[1,0] = F140M_a.T[4,9]

F212N_m50.T[0,0] = F212N_m50.T[5,9]
F212N_m50.T[1,0] = F212N_m50.T[4,9]

F212N_a.T[0,0] = F212N_a.T[5,9]
F212N_a.T[1,0] = F212N_a.T[4,9]

In [ ]:
plt.imshow(F140M_m50.T, cmap='jet')

In [ ]:
plt.imshow(F212N_m50.T, cmap='jet')

In [ ]:
np.save("../photometry/m82/F140M_m50.npy",F140M_m50)
np.save("../photometry/m82/F212N_m50.npy",F212N_m50)

np.save("../photometry/m82/F140M_a.npy",F140M_a)
np.save("../photometry/m82/F212N_a.npy",F212N_a)